In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations
from typing import Dict,List,Sequence
import random
import math

In [3]:
morse: Dict[str, str] = {'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.'}
chars: List[str] = list(morse.keys())

max_len: int = max(len(s) for s in morse.values())
print(max_len)

4


In [4]:
sym_to_vec : Dict[str, list] = {
    ".":([1.0, 0.0, 0.0]),
    "-":([0.0, 1.0, 0.0]),
    "_":([0.0, 0.0, 1.0]),
}
print(sym_to_vec['.'])
print(sym_to_vec['-'])
print(sym_to_vec["_"])



[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]


In [5]:
print(list(sym_to_vec.keys()))
print(list(sym_to_vec.values())) #Cool encoding sequence

['.', '-', '_']
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


In [6]:
def encode(seq:str) -> list:
    x : List[int] = []
    for i in range(max_len):
        sym: str = seq[i] if i < len(seq) else "_" #'.' → ['.', '_', '_', '_'] or '-...' → ['-', '.', '.', '.'] padding
        x.append(sym_to_vec[sym])
    return sum(x,[])
print(morse["A"])
print(encode(morse["A"])) #Testing  our function
print("dot,dash,padding,padding")


.-
[1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0]
dot,dash,padding,padding


In [7]:
def one_hot(i:int,n:int) -> list:
    y:list = [0.0] * n # create n number of zeroes list.
    y[i] = 1.0 # assign 1.0 to whichever number position i is.
    return y
for __ in range(4): #testing the matrix format
    print(one_hot(__,max_len))

[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]


In [8]:
# Build the full training input matrix
X = [encode(morse[c]) for c in chars]

# Build the target output matrix using manual one-hot encoding
Y = []
for i in range(len(chars)):
    one_hot_vector = [0] * len(chars)
    one_hot_vector[i] = 1
    Y.append(one_hot_vector)

## On to the neural net (Leaky)

In [9]:
class MLP:
    def __init__(self, n_in: int, n_hidden: int, n_out: int) -> None:
        self.W1: list = [[random.gauss(0, 0.4) for _ in range(n_hidden)] for _ in range(n_in)] # Nested lists for matrix creation
        self.b1 = [0.0] * n_hidden # Hidden-layer biases.
        self.W2 = [[random.gauss(0, 0.4) for _ in range(n_out)] for _ in range(n_hidden)] # Hidden-to-output weights.
        self.b2 = [0.0] * n_out # Output-layer biases.

    def leaky_relu(self, z: list, alpha: float = 0.01) -> list: # Leaky is more akin to nature
        return [[max(x, alpha * x) for x in row] for row in z]

    def dleaky_relu(self, z: list, alpha: float = 0.01) -> list:
        # Leaky ReLU helps prevent "dead neurons" by allowing a small gradient for negative inputs,
        # unlike sigmoid which suffers from vanishing gradients.
        if not z or not z[0]:
            return z

        if isinstance(z[0], (int, float)):
            # 1D list
            return [1 if x > 0 else alpha for x in z]
        else:
            # 2D list
            return [[1 if x > 0 else alpha for x in row] for row in z]

    def softmax(self, z: list) -> list: # Subtract list of z from its max value, exponentiate and divide by its sum(exp)
        max_val: list = max(z)
        shifted: list = [x - max_val for x in z] # Stabilize
        exps: list = [math.exp(x) for x in shifted] # Smooth
        total: int = sum(exps)
        return [x / total for x in exps] # Return list of probabilities

    def forward(self, X: list) -> list: # Forward pass
        # Perform matrix multiplication and add bias
        self.z1: list = [[0 for _ in range(len(self.W1[0]))] for _ in range(len(X))]
        for i in range(len(X)):             # For each row in X
            for j in range(len(self.W1[0])): # For each column in W1
                for k in range(len(X[0])):   # For each row in W1 (or column in X)
                    self.z1[i][j] += X[i][k] * self.W1[k][j]
                self.z1[i][j] += self.b1[j] # Adding the bias after the dot product
        self.a1: list = self.leaky_relu(self.z1)

        self.z2 = [[0 for _ in range(len(self.W2[0]))] for _ in range(len(self.a1))]
        for i in range(len(self.a1)):        # For each leaky relu row a1
            for j in range(len(self.W2[0])): # For each column in W2
                for k in range(len(self.W2)): # For each row in W2
                    self.z2[i][j] += self.a1[i][k] * self.W2[k][j]
                self.z2[i][j] += self.b2[j] # Adding the second bias after the dot product

        # FIX 1: Output layer must use softmax, not leaky_relu.
        # Softmax produces a probability distribution required by cross-entropy loss;
        # leaky_relu on the output layer produces unbounded values, making the loss meaningless.
        self.a2: list = [self.softmax(row) for row in self.z2]
        return self.a2

    def loss(self, Yhat: list, Y: list) -> float: # Cross-entropy loss for classification.
        EPS: float = 1e-12
        batch_size: int = len(Y)
        total_loss: float = 0.0
        for i in range(batch_size):
            sample_loss = 0.0
            for j in range(len(Y[i])):
                sample_loss += Y[i][j] * math.log(Yhat[i][j] + EPS)
            total_loss += -sample_loss
        return total_loss / batch_size # Return the mean loss

    def train_step(self, X: list, Y: list, lr: float = 0.01) -> float: # lr lowered from 0.7 — too high a rate causes divergence.
        n: int = len(X)
        Yhat: list = self.forward(X) # Forward: compute predictions

        dZ2: list = [[(Yhat[i][j] - Y[i][j]) / n for j in range(len(Y[0]))] for i in range(len(Y))] # Output-layer error signal for softmax + cross-entropy.

        # FIX 2 & 3: dW2 must be shaped [n_hidden, n_out], not [n, n].
        # len(self.W2) = n_hidden, len(dZ2[0]) = n_out.
        # The outer loops must also iterate over n_hidden and n_out, not batch size n.
        dW2: list = [[0.0] * len(dZ2[0]) for _ in range(len(self.W2))]
        for i in range(len(self.W2)):        # For each neuron in the hidden layer (row in a1, col in a1.T)
            for j in range(len(dZ2[0])):     # For each neuron in the output layer
                for k in range(len(dZ2)):    # For each sample in the batch
                    dW2[i][j] += self.a1[k][i] * dZ2[k][j] # a1.T is a1[k][i] instead of the original (i,k) — called transpose

        db2: list = [sum(column) for column in zip(*dZ2)] # Gradient for output biases.

        dA1: list = [[0.0] * len(self.W2) for _ in range(len(dZ2))]
        for i in range(len(dZ2)):            # For each sample in the batch
            for j in range(len(self.W2)):    # For each neuron in the hidden layer
                for k in range(len(dZ2[0])): # For each neuron in the output layer
                    dA1[i][j] += dZ2[i][k] * self.W2[j][k] # W2[j][k] is element (j,k) of W2.T

        # FIX 4: dleaky_relu must receive z1 (pre-activation), not a1 (post-activation).
        # The chain rule requires the derivative of the activation function at the point
        # before it was applied — using a1 gives the wrong gradient signal.
        # Element-wise multiplication of two 2D lists.
        dZ1 = [[dA1[i][j] * self.dleaky_relu(self.z1)[i][j]
                for j in range(len(dA1[0]))] for i in range(len(dA1))] # Hidden-layer error after relu derivative.

        # FIX 5 & 6: dW1 must be shaped [n_in, n_hidden], not [n, n].
        # len(X[0]) = n_in, len(dZ1[0]) = n_hidden.
        # The outer loops must also iterate over n_in and n_hidden, not batch size n.
        dW1: list = [[0.0] * len(dZ1[0]) for _ in range(len(X[0]))]
        for i in range(len(X[0])):           # For each feature in the input
            for j in range(len(dZ1[0])):     # For each neuron in the hidden layer
                for k in range(len(X)):      # For each sample in the batch
                    dW1[i][j] += X[k][i] * dZ1[k][j] # X[k][i] is element (i,k) of X.T

        db1: list = [sum(column) for column in zip(*dZ1)] # Gradient for hidden biases.

        # Update weights and biases by multiplying gradient with learning rate.
        # Update output weights (element-wise)
        self.W2 = [[self.W2[i][j] - lr * dW2[i][j] for j in range(len(self.W2[0]))] for i in range(len(self.W2))]

        # Update output biases
        self.b2 = [self.b2[j] - lr * db2[j] for j in range(len(self.b2))]

        # Update hidden weights
        self.W1 = [[self.W1[i][j] - lr * dW1[i][j] for j in range(len(self.W1[0]))] for i in range(len(self.W1))]

        # Update hidden biases
        self.b1 = [self.b1[j] - lr * db1[j] for j in range(len(self.b1))]

        return self.loss(Yhat, Y)

In [10]:
net: MLP = MLP(n_in=12, n_hidden=6, n_out=5)  # Create the network: 12 inputs, 6 hidden units, 5 outputs.

In [11]:
print(f"X shape: {len(X)} samples, each with {len(X[0])} features")
print(f"Y shape: {len(Y)} samples, each with {len(Y[0])} outputs")
print(f"Network input size: {net.W1.shape[0] if hasattr(net.W1, 'shape') else len(net.W1)}")  # For list-based
print(f"Network hidden size: {len(net.W1[0]) if net.W1 else 'unknown'}")

X shape: 5 samples, each with 12 features
Y shape: 5 samples, each with 5 outputs
Network input size: 12
Network hidden size: 6


In [12]:
print(f"X: {len(X)} samples, each sample has {len(X[0])} features")
print(f"W1: {len(net.W1)} rows (input features), {len(net.W1[0])} columns (hidden neurons)")
print(f"Expected input features from W1: {len(net.W1)}")
print(f"Actual input features in X: {len(X[0])}")

X: 5 samples, each sample has 12 features
W1: 12 rows (input features), 6 columns (hidden neurons)
Expected input features from W1: 12
Actual input features in X: 12


0.7 step_size although large yields the best final loss @ 5k iterations.

In [13]:
for epoch in range(5000):  # Train for a fixed number of epochs.
    loss: float = net.train_step(X, Y, lr=0.7)  # Update the weights using back-propagation.

In [14]:
pred: list = net.forward(X)  # Run the trained network on the training set.
labels: List[str] = [chars[max(range(len(p)), key=lambda i: p[i])] for p in pred]  # Find index of max probability  # Convert predicted probabilities to class labels.

In [15]:
print('final loss:', loss)  # Show the final training loss.
print('predictions:', labels)  # Show the predicted characters.

final loss: 0.0022565495515332185
predictions: ['A', 'B', 'C', 'D', 'E']
